# 03 · Stress sweeps, communication sensitivity and robustness profile
Table and figure numbers refer to manuscript draft v5, in which all tables are in Section 5 (there is no appendix). All figures are saved to `../figures/` at 600 dpi.

| Figure file | Illustrates (draft v5) | Style |
|---|---|---|
| `Table16_comm_matrix.png` | Table 16: energy under packet-loss and range sweeps | sensitivity matrix (% change from the base level) |
| `Table17_packet_loss.png` | Table 17: packet-loss sweep | annotated rank matrix |
| `Table18_comm_range.png` | Table 18: communication-range sweep | annotated rank matrix |
| `Table19_task_density.png` | Table 19: task-density sweep (draft Figure 9 is the same information) | annotated rank matrix |
| `Table20_uav_count.png` | Table 20: UAV-count sweep (draft Figure 10) | annotated rank matrix |
| `Table21_dropout.png` | Table 21: dropout summary (coverage, energy, stranded) | annotated rank matrix |
| `Table22_dropout.png` | Table 22: UAV-dropout sweep | annotated rank matrix |
| `Table23_deadline.png` | Table 23: deadline sweep | annotated rank matrix |
| `Table24_coupled.png` | Table 24: coupled-task sweep | annotated rank matrix |
| `Table25_obstacles.png` | Table 25: obstacle-density sweep (open and layered airspace) | annotated rank matrix |
| `Robustness_matrix_Tables16-25.png` | summary of Tables 16–25 | balloon fingerprint, all metrics, one panel per environment |


In [ ]:
import sys
sys.path.insert(0, "..")          # common.py lives in the package root
from common import *
%matplotlib inline

## Table 16 · communication sensitivity matrix
Energy per task of the seven non-trivial methods under packet-loss (0–60%) and range (6–50 m) sweeps. The colour shows the change relative to the base level of each sweep (20% loss, 12 m): green = less energy, red = more, saturating at ±30%. Flat rows (HRL-H, DMPC, Greedy) are immediately visible.

In [ ]:
cols = [("packet_loss", i) for i in range(4)] + [("comm_range", i) for i in range(4)]
labels = ["0%", "20%\n(base)", "40%", "60%", "6 m", "12 m\n(base)", "25 m", "50 m"]
base_idx = {"packet_loss": 1, "comm_range": 1}
fig, axs = plt.subplots(2, 2, figsize=(6.9, 4.7))
for k, e in enumerate(ENVS):
    ax = axs[k // 2, k % 2]
    V = np.array([[SW[s]["res"][e][m][i]["energy_wh_per_task_mean"] for s, i in cols] for m in SWEEP_METH], float)
    B = np.array([[V[r, base_idx[s] + (0 if s == "packet_loss" else 4)] for s, i in cols] for r in range(len(SWEEP_METH))])
    pct = V / B - 1
    S = np.clip(0.5 - pct / 0.6, 0, 1)              # lower energy -> greener
    annotated_matrix(ax, V, S, [lambda v: f"{v:.2f}"] * 8, [LAB[m] for m in SWEEP_METH], labels,
                     hl_row=len(SWEEP_METH) - 1, fs=5.6)
    ax.axvline(3.5, color="k", lw=1.0)
    ax.set_title(f"({'abcd'[k]}) {ENVN[e]}", fontsize=8, pad=22)
fig.subplots_adjust(left=0.085, right=0.995, top=0.90, bottom=0.10, wspace=0.20, hspace=0.50)
cax = fig.add_axes([0.28, 0.035, 0.44, 0.016])
cb = fig.colorbar(plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(-30, 30)), cax=cax, orientation="horizontal")
cb.set_ticks([-30, 0, 30])
cb.set_ticklabels(["+30% energy", "no change", "-30% energy"])
cb.ax.invert_xaxis()
cb.ax.tick_params(labelsize=6.4, length=0)
cb.outline.set_visible(False)
save(fig, "Table16_comm_matrix")


## Sweep matrices for Tables 17–25
Each figure uses the same layout as Tables 10 and 15: one panel per environment, methods as rows. Columns are grouped by metric — energy per completed task for every sweep level, then coverage for every level. Table 21 is the dropout summary (coverage, energy, stranded); Table 22 is the dropout sweep (energy, coverage), matching Tables 17–20 and 23–25. Colour is the rank of the method within each column (green = best, red = worst; grey = all values equal). Printed values are means over 10 seeds; ± std stays in the draft tables. Random is omitted so the colour scale separates the seven non-trivial methods (as in Table 16). Table 25 has two panels because the obstacle sweep covers only open and layered airspace.


In [ ]:
sweep_matrix("packet_loss", ["0%", "20%", "40%", "60%"], "Table17_packet_loss")
sweep_matrix("comm_range", ["6 m", "12 m", "25 m", "50 m"], "Table18_comm_range")
sweep_matrix("task_density", ["4", "8", "16", "32"], "Table19_task_density")
sweep_matrix("scalability", ["4", "8", "16"], "Table20_uav_count")
sweep_matrix("fault", ["0%", "25%", "50%"], "Table21_dropout", metrics=DROPOUT_SUMMARY_METRICS, fs=5.3)
sweep_matrix("fault", ["0%", "25%", "50%"], "Table22_dropout")
sweep_matrix("deadline", ["none", "200", "100"], "Table23_deadline")
sweep_matrix("coupled", ["single", "half:2", "all:2"], "Table24_coupled")
sweep_matrix("obstacles", ["0", "2", "4"], "Table25_obstacles")


## Robustness fingerprint per environment
One panel per environment. Each circle is one recorded metric, averaged at the hardest level of every stressor that includes that environment (obstacle density only in open and layered airspace). Colour and area are the rank among the seven methods (green/large = best, red/small = worst; grey = tied or non-directional). Printed values are those means. The bar **R** is the mean rank across directional metrics.


In [ ]:
robustness_fingerprint()
